# Part 1: Data Loading & Setup



## Q1. Spark Session & Data Loading
    Create a SparkSession
    Load sales_data.csv
    Infer schema and display schema


In [77]:
from pyspark.sql import SparkSession

# creating sparkSession
spark = SparkSession.builder\
            .appName("Aggregation-Window-Date-TimeStamp-Functions-Assignment")\
            .getOrCreate()


In [78]:
# loading sales_data.csv and infer schema
df = spark.read\
            .option("header",True)\
            .option("inferschema",True)\
            .csv('sales_data.csv')

# display schema
df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



In [79]:
from pyspark.sql.types import *
from pyspark.sql.functions import *

## Q2. Data Type Conversion
    Convert order_date to DateType
    Convert order_timestamp to TimestampType


In [80]:
# convert order_date to DateType
# convert order_timestamp to TimestampType
df = df.withColumn("order_date",col("order_date").cast(DateType()))\
        .withColumn("order_timestamp",col("order_timestamp").cast(TimestampType()))


df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product: string (nullable = true)
 |-- order_amount: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_timestamp: timestamp (nullable = true)



# Part 2: Aggregate Functions

## Q3. Overall Sales Metrics
    Calculate:

    Total sales amount (sum)
    Average order amount (avg)
    Maximum and minimum order amount


In [81]:
# total sales amount
df.select(sum("order_amount").alias("total_sales")).show()

# average order amount
df.select(avg("order_amount").alias("avg_order_amount")).show()

# maximum and minimum order amount
df.select(max("order_amount").alias("max_amount"),
            min("order_amount").alias("min_amount")
        ).show()

+-----------+
|total_sales|
+-----------+
|     359000|
+-----------+

+----------------+
|avg_order_amount|
+----------------+
|         44875.0|
+----------------+

+----------+----------+
|max_amount|min_amount|
+----------+----------+
|     80000|     20000|
+----------+----------+



## Q4. Region-wise Analysis:
    For each region, calculate:

    Total sales
    Average sales
    Order count

In [82]:
# total sales per region
df.groupBy("region")\
    .agg(sum("order_amount").alias("total_sales_per_region"))\
    .show()

# average sales per region
df.groupBy("region")\
    .agg(avg("order_amount").alias("average_sales_per_region"))\
    .show()

# order count per region
df.groupBy("region")\
    .agg(count("order_amount").alias("order_count_per_region"))\
    .show()

+------+----------------------+
|region|total_sales_per_region|
+------+----------------------+
| South|                102000|
|  East|                102000|
|  West|                 28000|
| North|                127000|
+------+----------------------+

+------+------------------------+
|region|average_sales_per_region|
+------+------------------------+
| South|                 51000.0|
|  East|                 51000.0|
|  West|                 28000.0|
| North|      42333.333333333336|
+------+------------------------+

+------+----------------------+
|region|order_count_per_region|
+------+----------------------+
| South|                     2|
|  East|                     2|
|  West|                     1|
| North|                     3|
+------+----------------------+



## Q5. Customer Count:
    Number of distinct customers using countDistinct()

In [83]:
# Number of distinct customers
df.select(count_distinct(col("customer_id")).alias("distinct_customers")).show()

+------------------+
|distinct_customers|
+------------------+
|                 4|
+------------------+



## Q6. Product-wise Aggregation:
    collect_list(order_amount)
    collect_set(order_amount)

In [84]:
# product wise collect_list, collect_set
df.groupBy("product").agg(
    collect_list("order_amount"),
    collect_set("order_amount")
).show(truncate=False)

+-------+--------------------------+-------------------------+
|product|collect_list(order_amount)|collect_set(order_amount)|
+-------+--------------------------+-------------------------+
|Laptop |[75000, 80000, 72000]     |[72000, 80000, 75000]    |
|Mobile |[30000, 28000, 32000]     |[28000, 30000, 32000]    |
|Tablet |[20000, 22000]            |[20000, 22000]           |
+-------+--------------------------+-------------------------+



# Part 3: Window Functions – Ranking


In [85]:
from pyspark.sql import Window

## Q7. Regional Ranking:
    For each region,
    Assign row_number() based on highest order amount
    Assign rank() and dense_rank()


In [86]:
# regional rankings (row_number, rank, dense_rank)
df.select(col("region"),
            row_number().over(Window.partitionBy(col("region")).orderBy(desc(col("order_amount")))).alias("row_number"),
            rank().over(Window.partitionBy(col("region")).orderBy(desc(col("order_amount")))).alias("rank"),
            dense_rank().over(Window.partitionBy(col("region")).orderBy(desc(col("order_amount")))).alias("dense_rank")
        ).show()

+------+----------+----+----------+
|region|row_number|rank|dense_rank|
+------+----------+----+----------+
|  East|         1|   1|         1|
|  East|         2|   2|         2|
| North|         1|   1|         1|
| North|         2|   2|         2|
| North|         3|   3|         3|
| South|         1|   1|         1|
| South|         2|   2|         2|
|  West|         1|   1|         1|
+------+----------+----+----------+



## Q8. Top Orders per Region
    Find the top 2 highest orders per region using window functions.


In [87]:
# top-2 highest orders per region
# if there are less than 2 records, it will only return those only (no padding of records)

df.select(
    col("region"),
    row_number().over(Window.partitionBy(col("region")).orderBy(desc(col("order_amount")))).alias("row_number")
).filter(col("row_number") <= 2).show()

+------+----------+
|region|row_number|
+------+----------+
|  East|         1|
|  East|         2|
| North|         1|
| North|         2|
| South|         1|
| South|         2|
|  West|         1|
+------+----------+



# Part 4: Window Functions – Analytical


## Q9. Order Comparison per Customer
    For each customer:
    Use lag() to show previous order amount
    Use lead() to show next order amount


In [88]:
df.select(
    col("customer_id"),
    lag(col("order_amount"),1,0).over(Window.partitionBy("customer_id").orderBy(col("order_timestamp"))).alias("prev_order_amount"),
    lead(col("order_amount"),1,0).over(Window.partitionBy("customer_id").orderBy(col("order_timestamp"))).alias("next_order_amount")    
).show()

+-----------+-----------------+-----------------+
|customer_id|prev_order_amount|next_order_amount|
+-----------+-----------------+-----------------+
|       C001|                0|            20000|
|       C001|            75000|            32000|
|       C001|            20000|                0|
|       C002|                0|            72000|
|       C002|            30000|                0|
|       C003|                0|            22000|
|       C003|            80000|                0|
|       C004|                0|                0|
+-----------+-----------------+-----------------+



## Q10. Running Total
    Calculate running total (cumulative sum) of order amount:
    Partition by customer_id
    Order by order_date

In [89]:
df.select(
    col("customer_id"),
    col("order_date"),
    col("order_amount"),
    sum(col("order_amount"))\
        .over(Window
              .partitionBy(col("customer_id"))
              .orderBy(col("order_date"))
              .rowsBetween(Window.unboundedPreceding,Window.currentRow)
            )
        .alias("running_total")
).show()

+-----------+----------+------------+-------------+
|customer_id|order_date|order_amount|running_total|
+-----------+----------+------------+-------------+
|       C001|2023-01-10|       75000|        75000|
|       C001|2023-02-05|       20000|        95000|
|       C001|2023-03-18|       32000|       127000|
|       C002|2023-01-12|       30000|        30000|
|       C002|2023-03-15|       72000|       102000|
|       C003|2023-02-20|       80000|        80000|
|       C003|2023-04-02|       22000|       102000|
|       C004|2023-03-01|       28000|        28000|
+-----------+----------+------------+-------------+



# Part 5: Window Functions – Aggregates


## Q11. Average Order per Customer
    Add a column avg_customer_order showing the average order amount per customer using window functions

In [90]:
df.select(
    col("customer_id"),
    avg(col("order_amount")).over(Window.partitionBy("customer_id")).alias("avg_customer_order")
).distinct().show()

+-----------+------------------+
|customer_id|avg_customer_order|
+-----------+------------------+
|       C001|42333.333333333336|
|       C002|           51000.0|
|       C003|           51000.0|
|       C004|           28000.0|
+-----------+------------------+



## Q12. Maximum Order per Region
Add a column max_region_order showing the maximum order amount within each region

In [91]:
df.select(
    col("region"),
    max(col("order_amount")).over(Window.partitionBy("customer_id")).alias("max_region_order")
).distinct().show()


+------+----------------+
|region|max_region_order|
+------+----------------+
|  West|           28000|
|  East|           80000|
| North|           75000|
| South|           72000|
+------+----------------+



## Part 6: Date & Timestamp Functions


## Q13. Date Components Extraction:
Year, Month, Day from order_date

In [92]:
df.select(
    col("order_date"),
    year(col("order_date")).alias("year"),
    month(col("order_date")).alias("month"),
    day(col("order_date")).alias("day")
).show()

+----------+----+-----+---+
|order_date|year|month|day|
+----------+----+-----+---+
|2023-01-10|2023|    1| 10|
|2023-01-12|2023|    1| 12|
|2023-02-05|2023|    2|  5|
|2023-02-20|2023|    2| 20|
|2023-03-01|2023|    3|  1|
|2023-03-15|2023|    3| 15|
|2023-03-18|2023|    3| 18|
|2023-04-02|2023|    4|  2|
+----------+----+-----+---+



## Q14. Date Difference:
    Days between today and order_date using datediff()

In [93]:
df.select(
    col("order_date"),
    datediff(current_date(),col("order_date")).alias("days_till_today")
).show()

+----------+---------------+
|order_date|days_till_today|
+----------+---------------+
|2023-01-10|           1170|
|2023-01-12|           1168|
|2023-02-05|           1144|
|2023-02-20|           1129|
|2023-03-01|           1120|
|2023-03-15|           1106|
|2023-03-18|           1103|
|2023-04-02|           1088|
+----------+---------------+



## Q15. Create new columns:
    order_year
    order_month
    order_week

In [94]:
# NOTE: here order_week does not specify what week to choose like 
# weekOfYear, or weekOfMonth, or anything else
# so we go for built-in function -> weekofyear

df.withColumn(
    "order_year",year(col("order_date"))
).withColumn(
    "order_month",month(col("order_date"))
).withColumn(
    "order_week",weekofyear(col("order_date"))
).show()

+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|order_year|order_month|order_week|
+--------+-----------+------+-------+------------+----------+-------------------+----------+-----------+----------+
|    1001|       C001| North| Laptop|       75000|2023-01-10|2023-01-10 10:15:00|      2023|          1|         2|
|    1002|       C002| South| Mobile|       30000|2023-01-12|2023-01-12 11:20:00|      2023|          1|         2|
|    1003|       C001| North| Tablet|       20000|2023-02-05|2023-02-05 09:10:00|      2023|          2|         5|
|    1004|       C003|  East| Laptop|       80000|2023-02-20|2023-02-20 14:45:00|      2023|          2|         8|
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|      2023|          3|         9|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 

## Q16. Date-based Filtering:
    Placed in March 2023

In [95]:
df.where((year(col("order_date")) == 2023) & (month(col("order_date")) == 3)).show()

+--------+-----------+------+-------+------------+----------+-------------------+
|order_id|customer_id|region|product|order_amount|order_date|    order_timestamp|
+--------+-----------+------+-------+------------+----------+-------------------+
|    1005|       C004|  West| Mobile|       28000|2023-03-01|2023-03-01 16:30:00|
|    1006|       C002| South| Laptop|       72000|2023-03-15|2023-03-15 18:25:00|
|    1007|       C001| North| Mobile|       32000|2023-03-18|2023-03-18 20:10:00|
+--------+-----------+------+-------+------------+----------+-------------------+



# Part 7: Real-World ETL Scenarios


## Q17. Monthly Sales Trend:
    Calculate monthly total sales (group by Year + Month)

In [96]:
df.withColumn("month",date_trunc("month",col("order_date")))\
    .groupBy("month")\
    .agg(sum("order_amount").alias("total_sales"))\
    .orderBy("month")\
    .show()

+-------------------+-----------+
|              month|total_sales|
+-------------------+-----------+
|2023-01-01 00:00:00|     105000|
|2023-02-01 00:00:00|     100000|
|2023-03-01 00:00:00|     132000|
|2023-04-01 00:00:00|      22000|
+-------------------+-----------+



## Q18. Customer Activity Analysis:
Identify customers who have placed more than 2 orders

In [97]:
df.groupBy("customer_id")\
        .agg(count("order_id").alias("total_orders"))\
        .where(col("total_orders") > 2)\
        .show()

+-----------+------------+
|customer_id|total_orders|
+-----------+------------+
|       C001|           3|
+-----------+------------+

